# Improvement

Six choices went into `src/`. This is each one and the measurement that settled it, in the
order I hit them. The cells import `data`, `model` and `evaluate` from `src/` rather than
repeating the code, so what runs here is what ships.

In [1]:
import logging
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = Path(f"{PROJECT_ROOT}/src")
sys.path.insert(0, str(SRC))

# src logs instead of printing, so route it to the notebook output.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

import numpy as np
import data
import model
import evaluate

In [2]:
texts, labels = data.load(data.default_data_path())
print(f"{len(texts)} rows after dropping exact repeats")
Counter(labels)

400 rows after dropping exact repeats


Counter({'general': 160,
         'account-access': 100,
         'transaction-dispute': 90,
         'fraud-report': 50})

## Group by template, not by row

The 400 rows are generated: a base template with slots filled in, plus an optional greeting
or sign-off. `data.template_key` peels the greetings and sign-offs and replaces the coin,
device and number slots with placeholders, so siblings of one template collapse to one key.
Grouping matters because a random split puts siblings on both sides of the line, and a
model can then score well by recognising a template it has already seen.

In [3]:
groups = data.groups(texts)
print(f"{len(texts)} rows -> {len(set(groups))} templates")

# Largest template families: these are the rows a random split would spread across folds.
Counter(groups).most_common(5)

400 rows -> 70 templates


[('what s the minimum amount i can buy of <coin>', 14),
 ('can you explain how to move <coin> to an external wallet', 14),
 ('is there an <device> for the <device> how do i get started', 13),
 ('how long do <coin> withdrawals usually take to process', 12),
 ('how do i enable price alerts for <coin>', 12)]

In [4]:
# Two rows from one family, and the key they share.
family = Counter(groups).most_common(1)[0][0]
for t in [t for t, g in zip(texts, groups) if g == family][:3]:
    print(repr(t))
print("\nkey:", repr(family))

"What's the minimum amount I can buy of Cardano?"
"Hey, What's the minimum amount I can buy of Ethereum? Please advise."
"Hello team, What's the minimum amount I can buy of Polygon?"

key: 'what s the minimum amount i can buy of <coin>'


Grouping strength is itself a decision. The three rows above differ only by coin and by a
greeting or sign-off, so a random split can train on the Cardano phrasing and test on the
Polygon one — that is the leak, and it is worth seeing concretely rather than assuming.
The largest families run 12-14 rows each, so a single template can span several folds if it
is not held together. If the grouped numbers below ever look suspiciously clean, check that
the key is still collapsing families like this one before crediting the model.

## Grouped CV is the protocol

Same model, same data, two splits. The random split reuses templates across the line; the
grouped split does not. The gap between the two rows is the entire size of the handover's
error: it is not that the baseline was measured badly by a few points, it is that a random
split on templated data cannot measure this model at all.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline

X, y, g = np.asarray(texts), np.asarray(labels), np.asarray(groups)

# The baseline's own recipe: word TF-IDF, unweighted logistic regression.
baseline = lambda: Pipeline([("tfidf", TfidfVectorizer()),
                             ("clf", LogisticRegression(max_iter=1000))])

def grouped(make_pipe, seeds=(0, 1, 2)):
    """Grouped-CV predictions, averaged over seeds because 70 groups is a small sample."""
    accs, f1s = [], []
    for seed in seeds:
        preds = np.empty(len(y), dtype=object)
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr, te in cv.split(X, y, groups=g):
            preds[te] = make_pipe().fit(X[tr], y[tr]).predict(X[te])
        accs.append(accuracy_score(y, preds))
        f1s.append(f1_score(y, preds, average="macro"))
    return float(np.mean(accs)), float(np.mean(f1s))

rand = cross_val_predict(baseline(), X, y, cv=5)
print(f"random 5-fold    acc {accuracy_score(y, rand):.4f}  macro-F1 {f1_score(y, rand, average='macro'):.4f}")
print("grouped 5-fold   acc {:.4f}  macro-F1 {:.4f}".format(*grouped(baseline)))

random 5-fold    acc 0.9850  macro-F1 0.9783


grouped 5-fold   acc 0.8233  macro-F1 0.7863


## Add character n-grams

`src/model.py` unions word (1-2) and character (2-4) TF-IDF. Character features let an
unseen phrasing match on shared substrings — "unauthoriz", "log in" against "login" — and
they absorb typos, which word features cannot do at all. This is the single largest source
of generalisation in the fix, so it is worth measuring on its own before any weighting.

In [6]:
from sklearn.pipeline import FeatureUnion

word_only = lambda: Pipeline([
    ("word", TfidfVectorizer(ngram_range=model.WORD_NGRAMS, min_df=1, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000, C=model.C)),
])

# Same classifier, features unioned: isolates what the char n-grams are worth.
word_char = lambda: model.build_pipeline()

for name, make in [("word only", word_only), ("word + char", word_char)]:
    acc, f1 = grouped(make)
    print(f"{name:<14} acc {acc:.4f}  macro-F1 {f1:.4f}")

word only      acc 0.8183  macro-F1 0.7925


word + char    acc 0.8750  macro-F1 0.8494


## Unbalanced class weights, skewed toward fraud

`class_weight="balanced"` computes inverse-frequency weights and stops, which equalises the
four routes and so assumes they cost the same. They do not: missing a fraud report loses
money, while over-flagging costs an analyst a few minutes. `model.fraud_first_weights`
therefore starts from inverse-frequency and pushes `fraud-report` past parity. The table
shows what each step buys and what it costs.

In [7]:
from sklearn.metrics import precision_score, recall_score

def fraud_scores(skew, seeds=(0, 1, 2, 3, 4)):
    """Macro-F1, fraud recall and fraud precision under grouped CV; skew=None means unweighted."""
    f1s, recalls, precisions = [], [], []
    for seed in seeds:
        if skew is None:
            preds = np.empty(len(y), dtype=object)
            cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr, te in cv.split(X, y, groups=g):
                preds[te] = model.build_pipeline().fit(X[tr], y[tr]).predict(X[te])
        else:
            preds = model._grouped_predict(X, y, g, seed, skew=skew)
        f1s.append(f1_score(y, preds, average="macro"))
        recalls.append(recall_score(y, preds, labels=[data.FRAUD], average="macro"))
        precisions.append(precision_score(y, preds, labels=[data.FRAUD],
                                          average="macro", zero_division=0))
    return np.mean(f1s), np.mean(recalls), np.mean(precisions)

In [8]:
print(f"{'weights':<18}{'macro-F1':>9}{'fraud rec':>11}{'fraud prec':>12}")
for name, skew in [("none", None), ("balanced", 1.0),
                   ("balanced x2.5", 2.5), ("balanced x3", 3.0)]:
    f1, rec, prec = fraud_scores(skew)
    print(f"{name:<18}{f1:9.4f}{rec:11.4f}{prec:12.4f}")

weights            macro-F1  fraud rec  fraud prec


none                 0.8464     0.6000      0.8429


balanced             0.8554     0.7760      0.7027


balanced x2.5        0.8488     0.9040      0.6112


balanced x3          0.8403     0.9200      0.5757


Read the recall column: unweighted, the model protects the larger classes and drops a
large share of fraud reports on the floor. Inverse-frequency weighting recovers most of
that, and skewing past it recovers the rest. Macro-F1 barely moves across the whole table,
which is the point — the choice is invisible in the headline metric and decisive in the one
that costs money.

## Derive the skew

`model.solve_skew` takes a fraud precision floor — how much analyst load the queue can absorb, 
the one genuine business input — and returns the largest skew still clearing it. Precision falls 
monotonically as skew rises, so the floor is the binding constraint and the weight falls out of measurement.

In [9]:
skew, measured = model.solve_skew(texts, labels, groups)
print(f"\nfloor {model.MIN_FRAUD_PRECISION:.2f} -> solved skew {skew} (FRAUD_SKEW = {model.FRAUD_SKEW})")

skew 1.00 -> fraud precision 0.7027


skew 1.25 -> fraud precision 0.6984


skew 1.50 -> fraud precision 0.6784


skew 1.75 -> fraud precision 0.6464


skew 2.00 -> fraud precision 0.6359


skew 2.50 -> fraud precision 0.6112


skew 3.00 -> fraud precision 0.5757



floor 0.60 -> solved skew 2.5 (FRAUD_SKEW = 2.5)


The skew is not a fixed property of the problem — it moves when the features change. Solving
it under both character ranges shows why that matters: the same 0.60 floor picks a different
weight depending on how good the features are. A hardcoded value would have gone stale here
without anything failing.

In [10]:
# Solve the skew under each char range; restore the shipped setting afterwards.
shipped_char = model.CHAR_NGRAMS
try:
    for char in [(2, 4), (2, 5)]:
        model.CHAR_NGRAMS = char
        solved, curve = model.solve_skew(texts, labels, groups)
        print(f"char={char} -> skew {solved:g}   "
              + "  ".join(f"{s:g}:{p:.3f}" for s, p in curve))
finally:
    model.CHAR_NGRAMS = shipped_char

skew 1.00 -> fraud precision 0.7027


skew 1.25 -> fraud precision 0.6984


skew 1.50 -> fraud precision 0.6784


skew 1.75 -> fraud precision 0.6464


skew 2.00 -> fraud precision 0.6359


...

skew 1.75 -> fraud precision 0.6090


skew 2.00 -> fraud precision 0.5933


skew 2.50 -> fraud precision 0.5681


skew 3.00 -> fraud precision 0.5356


char=(2, 5) -> skew 1.75   1:0.684  1.25:0.648  1.5:0.636  1.75:0.609  2:0.593  2.5:0.568  3:0.536


## Grid-search the rest, override one setting on purpose

`model.tune` grid-searches word range, char range and `C` under the same grouped CV,
averaged over five seeds because at 70 groups a single split picks its winner by noise.
The search's top config is kept except for `C`, which is held one step lower: the winning
`C` buys macro-F1 by shedding fraud recall, and recall is the guardrail here.

In [11]:
best, ranked = model.tune(texts, labels, groups)

print(f"\n{'macro-F1':>9}{'std':>8}   config")
for mean, std, params in ranked[:6]:
    print(f"{mean:9.4f}{std:8.4f}   word={params['features__word__ngram_range']}"
          f" char={params['features__char__ngram_range']} C={params['classifier__C']:g}")

best config {'classifier__C': 30.0, 'features__char__ngram_range': (2, 4), 'features__word__ngram_range': (1, 2)} at macro-F1 0.8578



 macro-F1     std   config
   0.8578  0.0038   word=(1, 2) char=(2, 4) C=30
   0.8572  0.0117   word=(1, 1) char=(2, 4) C=30
   0.8548  0.0083   word=(1, 1) char=(2, 4) C=10
   0.8537  0.0091   word=(1, 2) char=(2, 4) C=10
   0.8490  0.0123   word=(1, 1) char=(2, 5) C=30
   0.8485  0.0108   word=(1, 1) char=(2, 5) C=10


In [12]:
# What the top-ranked C costs on the guardrail metric.
for c in (10.0, 30.0):
    params = {**best, "classifier__C": c}
    recalls = []
    for seed in model.TUNE_SEEDS:
        preds = model._grouped_predict(X, y, g, seed, params=params)
        recalls.append(recall_score(y, preds, labels=[data.FRAUD], average="macro"))
    print(f"C={c:<5g} fraud recall {np.mean(recalls):.4f}")

print(f"\nshipped: C={model.C:g}, char={model.CHAR_NGRAMS}, word={model.WORD_NGRAMS}")

C=10    fraud recall 0.9040


C=30    fraud recall 0.8760

shipped: C=10, char=(2, 4), word=(1, 2)


`evaluate.report` is the shipped scorer: grouped CV over three seeds, macro-F1 as the
headline, fraud recall called out as the release guardrail, and both split protocols shown
side by side so the leakage gap stays visible in every run. Weights are derived per fold
from the training rows only — deriving them once from all 400 labels would leak the class
distribution into every fold.

In [13]:
scores = evaluate.report(data.default_data_path())

400 messages, 70 intent templates


grouped 5-fold over seeds [0, 1, 2]


macro-F1             0.8444 +/- 0.0092


fraud-report recall  0.9000 +/- 0.0163   <- release guardrail


fraud-report prec.   0.6071            <- floor bounds analyst load


random split macro-F1  1.0000  (what the handover measured)


grouped     macro-F1  0.8426  (what generalises)


per-class breakdown
                     precision    recall  f1-score   support

     account-access       0.95      0.81      0.88       100
       fraud-report       0.60      0.92      0.72        50
            general       0.93      0.84      0.89       160
transaction-dispute       0.87      0.90      0.89        90

           accuracy                           0.86       400
          macro avg       0.84      0.87      0.84       400
       weighted avg       0.88      0.86      0.86       400



confusion, rows true cols predicted ['account-access', 'fraud-report', 'general', 'transaction-dispute']
[[ 81  10   9   0]
 [  1  46   1   2]
 [  0  15 135  10]
 [  3   6   0  81]]


Fraud recall near 0.90 at ~0.61 precision is the trade the floor was set to buy: roughly
two fraud reports in five flagged items are false alarms, and the queue absorbs that in
exchange for catching nine in ten. Macro-F1 around 0.84 against 1.0000 on a random split
is the honest number — the drop is what the grouping bought us the ability to see.